# Hurricane Melissa Recovery Spatial Map: Months 5-6

This notebook maps recovery by months 5-6 for benefit-providing NbS areas that showed early post-event damage in months 1-2. It is designed as a companion to the hurricane exposure and NDVI response panel. River-flood forest restoration benefit pixels are shown on the river avoided-EAD grid; mangrove recovery is shown at patch level using mean recovery across damaged pixels within each benefit-providing patch.

In [ ]:
from pathlib import Path
import sys

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
robyn_library_path = base_path / "robyns_libraries"
if str(robyn_library_path) not in sys.path:
    sys.path.append(str(robyn_library_path))

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import Robyn_paper_2_defs
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from rasterio.warp import Resampling, reproject

Robyn_paper_2_defs.set_nature_style()
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Paths And Constants

In [ ]:
paper2_path = base_path / "dphil_paper_2"
paper3_path = base_path / "dphil_paper_3"
common_path = base_path / "dphil_common_cross_cutting"

output_dir = (
    paper3_path
    / "results"
    / "threats"
    / "hurricane_melissa_damage"
    / "recovery"
    / "recovery_spatial_map_months5_6"
)
output_dir.mkdir(parents=True, exist_ok=True)

jamaica_boundary_path = common_path / "common_incoming_data" / "boundaries" / "jamaica.gpkg"
mangrove_patches_path = paper3_path / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
mangrove_patch_recovery_path = (
    paper3_path
    / "results"
    / "threats"
    / "hurricane_melissa_damage"
    / "recovery"
    / "mangrove_ndvi_recovery_months1_6"
    / "mangrove_recovery_patch_summary.csv"
)
river_ead_min_path = paper2_path / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
river_ead_max_path = paper2_path / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
ndvi_before_path = paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
ndvi_months_1_2_path = paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
ndvi_months_5_6_path = paper3_path / "inputs" / "ndvi" / "HLS_masked_NDVI_months5to6_after_epsg3448_2026-03-01_to_2026-04-29.tif"
storm_track_path = paper3_path / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_lin.shp"
wind_swath_path = paper3_path / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_windswath.shp"

map_crs = "EPSG:3448"
figure_dpi = 300
jmd_to_usd = 1.0 / 150.0
relative_baseline_min = 0.20
relative_damage_threshold = -0.10
wind_threshold_order = [34.0, 50.0, 64.0]

for input_path in [
    jamaica_boundary_path,
    mangrove_patches_path,
    mangrove_patch_recovery_path,
    river_ead_min_path,
    river_ead_max_path,
    ndvi_before_path,
    ndvi_months_1_2_path,
    ndvi_months_5_6_path,
    storm_track_path,
    wind_swath_path,
]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

output_dir

## Helper Functions

In [ ]:
def clean_geometries(geodataframe: gpd.GeoDataFrame, target_crs: str) -> gpd.GeoDataFrame:
    """Reproject, repair, and remove empty geometries."""
    clean_geodataframe = geodataframe.to_crs(target_crs)
    clean_geodataframe = clean_geodataframe[clean_geodataframe.geometry.notna()].copy()
    clean_geodataframe = clean_geodataframe[~clean_geodataframe.geometry.is_empty].copy()
    clean_geodataframe["geometry"] = clean_geodataframe.geometry.make_valid()
    clean_geodataframe = clean_geodataframe[~clean_geodataframe.geometry.is_empty].copy()
    return clean_geodataframe


def read_noaa_lonlat_layer(path: Path, target_crs: str) -> gpd.GeoDataFrame:
    """Read NOAA best-track layers as lon/lat and reproject to the map CRS."""
    noaa_geodataframe = gpd.read_file(path)
    noaa_geodataframe = noaa_geodataframe.set_crs("EPSG:4326", allow_override=True)
    return clean_geometries(noaa_geodataframe, target_crs)


def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD raster, convert JMD to USD, and retain only positive pixels."""
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * jmd_to_usd
    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan
    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")
    return ead_array, profile


def reproject_continuous_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    """Reproject a continuous raster to the river restoration-benefit grid."""
    destination = np.full((reference_profile["height"], reference_profile["width"]), np.nan, dtype="float32")
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def valid_ndvi(ndvi_array: np.ndarray) -> np.ndarray:
    """Return valid NDVI cells on the expected -1 to 1 range."""
    return np.isfinite(ndvi_array) & (ndvi_array >= -1.0) & (ndvi_array <= 1.0)


def classify_recovery_fraction_pct(recovery_fraction_pct: float) -> int:
    """Classify recovery fraction percentage for mapping."""
    if not np.isfinite(recovery_fraction_pct):
        return 1
    if recovery_fraction_pct <= 0:
        return 2
    if recovery_fraction_pct < 50:
        return 3
    if recovery_fraction_pct < 100:
        return 4
    return 5


def add_standard_jamaica_map_furniture(axis: plt.Axes, boundary_geodataframe: gpd.GeoDataFrame) -> None:
    """Add standard upper-right Jamaica map furniture using shared helpers."""
    scale_bar_point = Robyn_paper_2_defs.add_scale_bar(
        axis,
        boundary_geodataframe,
        where="right-top",
        pad=0.07,
        length_km=20,
        max_frac=0.22,
        lw=0.5,
        tick_h_frac=0.010,
        fs_lab=5.5,
        fs_unit=5.5,
        unit_text="km",
    )
    if scale_bar_point is None:
        return
    center_data_x, center_data_y = scale_bar_point
    center_axes_x, center_axes_y = axis.transAxes.inverted().transform(
        axis.transData.transform((center_data_x, center_data_y))
    )
    Robyn_paper_2_defs.add_north_arrow_axes(
        axis,
        center_axes_x,
        center_axes_y,
        size_frac=0.060,
        gap_frac=0.030,
        shaft_w_frac=0.10,
        head_w_frac=0.32,
        head_h_frac=0.55,
        fs=5.8,
        lw=0.5,
    )


def set_jamaica_extent(axis: plt.Axes, boundary_geodataframe: gpd.GeoDataFrame) -> None:
    """Apply consistent Jamaica map extent with enough upper-right room for map furniture."""
    minimum_x, minimum_y, maximum_x, maximum_y = boundary_geodataframe.total_bounds
    map_width = maximum_x - minimum_x
    map_height = maximum_y - minimum_y
    axis.set_xlim(minimum_x - map_width * 0.025, maximum_x + map_width * 0.025)
    axis.set_ylim(minimum_y - map_height * 0.11, maximum_y + map_height * 0.10)
    axis.set_axis_off()


def save_figure(figure: plt.Figure, stem: str) -> list[Path]:
    """Save a figure to PNG, PDF, and SVG."""
    output_paths = []
    for suffix in ["png", "pdf", "svg"]:
        output_path = output_dir / f"{stem}.{suffix}"
        figure.savefig(output_path, bbox_inches="tight", facecolor="white", dpi=figure_dpi)
        output_paths.append(output_path)
    return output_paths

## Build Recovery Classes

In [ ]:
river_ead_min_usd, river_profile = read_positive_ead_usd(river_ead_min_path)
river_ead_max_usd, _ = read_positive_ead_usd(river_ead_max_path, river_profile)
river_transform = river_profile["transform"]
river_shape = (river_profile["height"], river_profile["width"])
river_pixel_area_ha = abs(river_transform.a * river_transform.e) / 10_000.0
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)
river_left, river_bottom, river_right, river_top = rasterio.transform.array_bounds(
    river_profile["height"],
    river_profile["width"],
    river_transform,
)

ndvi_before = reproject_continuous_to_reference(ndvi_before_path, river_profile)
ndvi_months_1_2 = reproject_continuous_to_reference(ndvi_months_1_2_path, river_profile)
ndvi_months_5_6 = reproject_continuous_to_reference(ndvi_months_5_6_path, river_profile)

full_series_mask = (
    river_benefit_mask
    & valid_ndvi(ndvi_before)
    & valid_ndvi(ndvi_months_1_2)
    & valid_ndvi(ndvi_months_5_6)
    & (ndvi_before >= relative_baseline_min)
)
relative_change_months_1_2 = np.full(river_shape, np.nan, dtype="float32")
relative_change_months_1_2[full_series_mask] = (
    ndvi_months_1_2[full_series_mask] - ndvi_before[full_series_mask]
) / ndvi_before[full_series_mask]
river_damaged_mask = full_series_mask & (relative_change_months_1_2 <= relative_damage_threshold)
initial_ndvi_drop = ndvi_before - ndvi_months_1_2
recovery_fraction = np.full(river_shape, np.nan, dtype="float32")
valid_drop_mask = river_damaged_mask & (initial_ndvi_drop > 0)
recovery_fraction[valid_drop_mask] = (
    ndvi_months_5_6[valid_drop_mask] - ndvi_months_1_2[valid_drop_mask]
) / initial_ndvi_drop[valid_drop_mask]

river_recovery_class = np.zeros(river_shape, dtype="uint8")
river_recovery_class[river_benefit_mask & ~river_damaged_mask] = 1
river_recovery_class[river_damaged_mask & (recovery_fraction <= 0)] = 2
river_recovery_class[river_damaged_mask & (recovery_fraction > 0) & (recovery_fraction < 0.5)] = 3
river_recovery_class[river_damaged_mask & (recovery_fraction >= 0.5) & (recovery_fraction < 1.0)] = 4
river_recovery_class[river_damaged_mask & (recovery_fraction >= 1.0)] = 5
river_recovery_class = np.ma.masked_where(river_recovery_class == 0, river_recovery_class)

class_labels = {
    1: "Benefit area not in damaged recovery subset",
    2: "No improvement/further decline",
    3: "<50% of initial NDVI loss recovered",
    4: "50-<100% of initial NDVI loss recovered",
    5: "Recovered to/beyond pre-event NDVI",
}
class_colors = {
    1: "#d9d9d9",
    2: "#8c2d04",
    3: "#d73027",
    4: "#fdae61",
    5: "#1a9850",
}
recovery_cmap = ListedColormap([class_colors[class_value] for class_value in range(1, 6)])
recovery_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], recovery_cmap.N)

print(f"River recovery damaged area: {river_damaged_mask.sum() * river_pixel_area_ha:,.1f} ha")

In [ ]:
jamaica_boundary = clean_geometries(gpd.read_file(jamaica_boundary_path), map_crs)
storm_track = read_noaa_lonlat_layer(storm_track_path, map_crs)
wind_swath = read_noaa_lonlat_layer(wind_swath_path, map_crs)
wind_swath_by_threshold = wind_swath.dissolve(by="RADII", as_index=False)

mangrove_patches = clean_geometries(gpd.read_file(mangrove_patches_path), map_crs)
mangrove_patch_recovery = pd.read_csv(mangrove_patch_recovery_path)
mangrove_patches["Mangrove_ID"] = mangrove_patches["ID"].astype(int)
mangrove_recovery_patches = mangrove_patches.merge(mangrove_patch_recovery, on="Mangrove_ID", how="inner")
mangrove_recovery_patches["recovery_class"] = 1
damaged_mangrove_patch_mask = mangrove_recovery_patches["damaged_months_1_2_area_ha"].fillna(0) > 0
mangrove_recovery_patches.loc[damaged_mangrove_patch_mask, "recovery_class"] = mangrove_recovery_patches.loc[
    damaged_mangrove_patch_mask,
    "damaged_mean_recovery_fraction_of_initial_drop_by_months_5_6_pct",
].apply(classify_recovery_fraction_pct)

wind_line_colors = {34.0: "#6bb6ff", 50.0: "#2585d9", 64.0: "#0057b8"}

print(f"Mangrove benefit patches: {len(mangrove_recovery_patches):,}")
print(f"Damaged mangrove benefit patches: {int(damaged_mangrove_patch_mask.sum()):,}")

## Summaries

In [ ]:
river_summary_rows = []
for class_value, class_label in class_labels.items():
    class_mask = np.asarray(river_recovery_class.filled(0) == class_value)
    river_summary_rows.append(
        {
            "ecosystem": "River forest restoration",
            "recovery_class": class_label,
            "pixel_count": int(class_mask.sum()),
            "area_ha": class_mask.sum() * river_pixel_area_ha,
            "share_of_damaged_area_pct": (
                class_mask.sum() / river_damaged_mask.sum() * 100 if class_value > 1 and river_damaged_mask.sum() else np.nan
            ),
        }
    )

mangrove_summary_rows = []
for class_value, class_label in class_labels.items():
    patch_subset = mangrove_recovery_patches[mangrove_recovery_patches["recovery_class"].eq(class_value)]
    mangrove_summary_rows.append(
        {
            "ecosystem": "Mangroves",
            "recovery_class": class_label,
            "patch_count": len(patch_subset),
            "damaged_area_ha": patch_subset["damaged_months_1_2_area_ha"].fillna(0).sum(),
            "share_of_damaged_area_pct": (
                patch_subset["damaged_months_1_2_area_ha"].fillna(0).sum()
                / mangrove_recovery_patches["damaged_months_1_2_area_ha"].fillna(0).sum()
                * 100
                if class_value > 1 else np.nan
            ),
        }
    )

river_spatial_summary = pd.DataFrame(river_summary_rows)
mangrove_spatial_summary = pd.DataFrame(mangrove_summary_rows)
display(river_spatial_summary.round(2))
display(mangrove_spatial_summary.round(2))

## Spatial Map

In [ ]:
figure, axis = plt.subplots(figsize=(180 / 25.4, 92 / 25.4), dpi=figure_dpi)

jamaica_boundary.plot(ax=axis, facecolor="white", edgecolor="none", zorder=0)
axis.imshow(
    river_recovery_class,
    extent=(river_left, river_right, river_bottom, river_top),
    origin="upper",
    cmap=recovery_cmap,
    norm=recovery_norm,
    interpolation="nearest",
    zorder=2,
)

for class_value in range(1, 6):
    patch_subset = mangrove_recovery_patches[mangrove_recovery_patches["recovery_class"].eq(class_value)]
    if not patch_subset.empty:
        patch_subset.plot(
            ax=axis,
            facecolor=class_colors[class_value],
            edgecolor="#08306b" if class_value > 1 else "#7f7f7f",
            linewidth=0.20 if class_value > 1 else 0.08,
            alpha=0.95 if class_value > 1 else 0.65,
            zorder=5 if class_value > 1 else 4,
        )

for wind_threshold in wind_threshold_order:
    wind_threshold_geodataframe = wind_swath_by_threshold[wind_swath_by_threshold["RADII"].astype(float).eq(wind_threshold)]
    if not wind_threshold_geodataframe.empty:
        wind_threshold_geodataframe.boundary.plot(
            ax=axis,
            color=wind_line_colors[wind_threshold],
            linewidth=0.75 if wind_threshold < 64 else 0.95,
            zorder=7,
        )
storm_track.plot(ax=axis, color="black", linewidth=0.85, zorder=8)
jamaica_boundary.boundary.plot(ax=axis, color="black", linewidth=0.55, zorder=9)
set_jamaica_extent(axis, jamaica_boundary)
add_standard_jamaica_map_furniture(axis, jamaica_boundary)
axis.set_title("Months 5-6 recovery in benefit-providing NbS damaged in months 1-2", pad=2)

legend_handles = [
    mpatches.Patch(facecolor=class_colors[class_value], edgecolor="none", label=class_labels[class_value])
    for class_value in range(1, 6)
]
legend_handles.extend(
    [
        Line2D([0], [0], color=wind_line_colors[34.0], linewidth=0.8, label="34 kt wind threshold"),
        Line2D([0], [0], color=wind_line_colors[50.0], linewidth=0.8, label="50 kt wind threshold"),
        Line2D([0], [0], color=wind_line_colors[64.0], linewidth=1.0, label="64 kt wind threshold"),
        Line2D([0], [0], color="black", linewidth=0.85, label="Melissa track"),
    ]
)
axis.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.025),
    ncol=3,
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="#cfcfcf",
    handlelength=1.8,
    columnspacing=0.9,
    labelspacing=0.45,
)
figure_paths = save_figure(figure, "hurricane_melissa_recovery_spatial_map_months5_6")
plt.show()

figure_paths

## Save Outputs

In [ ]:
river_summary_path = output_dir / "river_forest_recovery_spatial_summary_months5_6.csv"
mangrove_summary_path = output_dir / "mangrove_recovery_spatial_summary_months5_6.csv"
metadata_path = output_dir / "hurricane_melissa_recovery_spatial_map_months5_6_metadata.csv"

river_spatial_summary.to_csv(river_summary_path, index=False)
mangrove_spatial_summary.to_csv(mangrove_summary_path, index=False)
metadata = pd.DataFrame(
    [
        {"name": "pre_event_ndvi", "value": str(ndvi_before_path)},
        {"name": "early_post_event_damage_state", "value": str(ndvi_months_1_2_path)},
        {"name": "recovery_window", "value": str(ndvi_months_5_6_path)},
        {"name": "damage_definition", "value": "relative NDVI change months 1-2 vs pre-event <= -0.10, baseline NDVI >= 0.20"},
        {"name": "recovery_fraction_definition", "value": "(NDVI_months_5_6 - NDVI_months_1_2) / (NDVI_pre_event - NDVI_months_1_2)"},
        {"name": "river_map_unit", "value": "river-flood forest restoration benefit pixels on river avoided-EAD grid"},
        {"name": "mangrove_map_unit", "value": "benefit-providing mangrove patches, classified by mean recovery across damaged pixels"},
        {"name": "north_arrow_and_scale_bar", "value": "shared Robyn_paper_2_defs helpers, right-top placement"},
    ]
)
metadata.to_csv(metadata_path, index=False)

for saved_path in [river_summary_path, mangrove_summary_path, metadata_path, *figure_paths]:
    print(saved_path)